In [496]:
import numpy as np
import pandas as pd

In [497]:
train_file=input(" Enter the file name : ")

 Enter the file name : kdd_train_prep2.csv


In [498]:
test_file=input(" Enter the file name : ")

 Enter the file name : kdd_test_prep2.csv


In [499]:
kdd_train= pd.read_csv(train_file)
kdd_test= pd.read_csv(test_file)

In [500]:
kdd_train.head()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH,attack,label_2,label_5
0,0.0,3.558064e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal
1,0.0,1.057999e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal
2,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,neptune,attack,DoS
3,0.0,1.681203e-07,6.223962e-06,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal
4,0.0,1.442067e-07,3.206260e-07,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal


In [501]:
label_list=['attack', 'label_2','label_5']
kdd_label=kdd_train[label_list]
test_label=kdd_test[label_list]

In [502]:
kdd_train['label_2'].value_counts()

normal    67343
attack    58630
Name: label_2, dtype: int64

In [503]:
kdd_test['label_2'].value_counts()

attack    12833
normal     9711
Name: label_2, dtype: int64

In [504]:
kdd_label['Class']=kdd_train['label_2'].apply(lambda x: 1 if x=='normal' else -1)
test_label['Class']=kdd_test['label_2'].apply(lambda x: 1 if x=='normal' else -1)


/mnt/disks/user/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.
/mnt/disks/user/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  


In [505]:
kdd_train.drop(label_list, axis = 1, inplace = True)
kdd_test.drop(label_list, axis = 1, inplace = True)


In [506]:
kdd_train.shape

(125973, 113)

In [507]:
kdd_test.shape

(22544, 107)

In [508]:
kdd_label.head()

,attack,label_2,label_5,Class
0,normal,normal,Normal,1
1,normal,normal,Normal,1
2,neptune,attack,DoS,-1
3,normal,normal,Normal,1
4,normal,normal,Normal,1


In [509]:
y_train=kdd_label['Class']
y_train.shape

(125973,)

### Anomaly Detection

In [510]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import make_scorer, f1_score
from sklearn import model_selection

In [537]:
from sklearn.ensemble import IsolationForest
import time

start_time = time.time()
clf=IsolationForest(n_estimators=155, max_samples=5, contamination=float(.46), \
                        max_features=9, bootstrap=False, n_jobs=-1, random_state=42, verbose=0)
clf.fit(kdd_train)
pred = clf.predict(kdd_train)
print("\n\nRun Time ->","--- %s seconds ---" % (time.time() - start_time))




Run Time -> --- 9.419333934783936 seconds ---


In [538]:
kdd_label['Anomaly']=pred

/mnt/disks/user/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


In [539]:
kdd_train.shape

(125973, 113)

In [540]:
kdd_label['Class'].value_counts()

 1    67343
-1    58630
Name: Class, dtype: int64

In [541]:
kdd_label['Anomaly'].value_counts()

 1    68011
-1    57962
Name: Anomaly, dtype: int64

In [542]:
kdd_label.head()

,attack,label_2,label_5,Class,Anomaly
0,normal,normal,Normal,1,1
1,normal,normal,Normal,1,-1
2,neptune,attack,DoS,-1,-1
3,normal,normal,Normal,1,1
4,normal,normal,Normal,1,-1


In [543]:
print(pd.crosstab(kdd_label['label_2'], kdd_label['Anomaly']),"\n")


Anomaly     -1      1
label_2              
attack   36149  22481
normal   21813  45530 



In [544]:
print(pd.crosstab(kdd_label['label_2'], kdd_label['Class']),"\n")


Class       -1      1
label_2              
attack   58630      0
normal       0  67343 



In [545]:
import sklearn.metrics


In [546]:
print(sklearn.metrics.classification_report(kdd_label['Class'], kdd_label['Anomaly']))

             precision    recall  f1-score   support

         -1       0.62      0.62      0.62     58630
          1       0.67      0.68      0.67     67343

avg / total       0.65      0.65      0.65    125973



In [547]:
y_pred=kdd_label['Anomaly']


In [548]:
def metrics_df(y_train, y_pred):
            print("Confusion Matrix :\n",sklearn.metrics.confusion_matrix(y_train, y_pred))

            print("Accuracy : \t",sklearn.metrics.accuracy_score(y_train, y_pred,normalize=True))

            print("F1_score : \t",sklearn.metrics.f1_score(y_train, y_pred))

            print("Precision : \t",sklearn.metrics.precision_score(y_train, y_pred))

            print("Recall : \t",sklearn.metrics.recall_score(y_train, y_pred))

            print("ROC_AUC : \t",sklearn.metrics.roc_auc_score(y_train, y_pred))      

            print("Normalized MI: \t",sklearn.metrics.normalized_mutual_info_score(y_train, y_pred)) 
            print("Adj_Rand_score: \t",sklearn.metrics.adjusted_rand_score(y_train, y_pred))
            print("HomogenityScore: \t",sklearn.metrics.homogeneity_score(y_train, y_pred))   
            
            confusion=sklearn.metrics.confusion_matrix(y_train, y_pred)
            TP = confusion[1,1] # true positive 
            TN = confusion[0,0] # true negatives
            FP = confusion[0,1] # false positives
            FN = confusion[1,0] # false negatives


            # Let's see the sensitivity of our logistic regression model
            print('Sensitivity/Recall :  \t {}'.format(TP / float(TP+FN)))
            # Let us calculate specificity
            print('Specificity: \t {}'.format(TN / float(TN+FP)))
            # Let us calculate specificity
            print('Precision: \t {}'.format(TP / float(TP+FP)))
            # Calculate false postive rate - predicting churn when customer does not have churned
            print('Accuracy:\t {}'.format((TP + TN)/ float(TP + TN + FP + FN)))

            print('Detection Rate:\t {}'.format(TP/ float(TP + TN + FP + FN)))

            print('False Positive Rate: \t{}'.format(FP/ float(TN+FP)))
            #False Negative
            print('False Negative Rate: \t {}'.format(FN/ float(TP+FN)))
            ## Misclassification Rtae : FP + FN / TP + TN + FP + FN
            print('Missclassification Rate: \t {}'.format((FP + FN)/ float(TP + TN + FP + FN)))
            # positive predictive value 
            print('Positive predictive value: \t {}'.format(TP / float(TP+FP)))
            # Negative predictive value
            print('Negative Predictive value: \t{}'.format(TN / float(TN+ FN)))


In [549]:
metrics_df(y_train, y_pred)

Confusion Matrix :
 [[36149 22481]
 [21813 45530]]
Accuracy : 	 0.6483849713827566
F1_score : 	 0.6727544069624836
Precision : 	 0.6694505300613136
Recall : 	 0.6760910562345009
ROC_AUC : 	 0.6463262717638478
Normalized MI: 	 0.06294334576929744
Adj_Rand_score: 	 0.08803747816350127
HomogenityScore: 	 0.06290728535128196
Sensitivity/Recall :  	 0.6760910562345009
Specificity: 	 0.6165614872931946
Precision: 	 0.6694505300613136
Accuracy:	 0.6483849713827566
Detection Rate:	 0.3614266549181174
False Positive Rate: 	0.3834385127068054
False Negative Rate: 	 0.323908943765499
Missclassification Rate: 	 0.35161502861724336
Positive predictive value: 	 0.6694505300613136
Negative Predictive value: 	0.6236672302543046


### Tets Data Evaluation

In [550]:
from sklearn.ensemble import IsolationForest
import time

start_time = time.time()
clf=IsolationForest(n_estimators=155, max_samples=10, contamination=float(.46), \
                        max_features=9, bootstrap=False, n_jobs=-1, random_state=42, verbose=0)
clf.fit(kdd_test)
test_pred = clf.predict(kdd_test)
print("\n\nRun Time ->","--- %s seconds ---" % (time.time() - start_time))




Run Time -> --- 2.4368391036987305 seconds ---


In [551]:
test_label['pred']=test_pred

/mnt/disks/user/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


In [552]:
y_test=test_label['Class']
metrics_df(y_test,test_pred)

Confusion Matrix :
 [[8261 4572]
 [2109 7602]]
Accuracy : 	 0.7036462029808375
F1_score : 	 0.6947224126113777
Precision : 	 0.6244455396747166
Recall : 	 0.7828236021007106
ROC_AUC : 	 0.7132773040504332
Normalized MI: 	 0.13660121290670532
Adj_Rand_score: 	 0.1657485890008706
HomogenityScore: 	 0.13724070385472642
Sensitivity/Recall :  	 0.7828236021007106
Specificity: 	 0.6437310060001559
Precision: 	 0.6244455396747166
Accuracy:	 0.7036462029808375
Detection Rate:	 0.3372072391767211
False Positive Rate: 	0.35626899399984413
False Negative Rate: 	 0.21717639789928947
Missclassification Rate: 	 0.29635379701916253
Positive predictive value: 	 0.6244455396747166
Negative Predictive value: 	0.7966248794599807


In [553]:
print(pd.crosstab(y_test,test_pred),"\n")


col_0    -1     1
Class            
-1     8261  4572
 1     2109  7602 



In [554]:
print(pd.crosstab(test_label['label_2'],test_pred),"\n")

col_0      -1     1
label_2            
attack   8261  4572
normal   2109  7602 



In [555]:
kdd_train.head()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0.0,3.558064e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,1.057999e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.0,1.681203e-07,6.223962e-06,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,1.442067e-07,3.206260e-07,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [556]:
kdd_label.head()

,attack,label_2,label_5,Class,Anomaly
0,normal,normal,Normal,1,1
1,normal,normal,Normal,1,-1
2,neptune,attack,DoS,-1,-1
3,normal,normal,Normal,1,1
4,normal,normal,Normal,1,-1


In [557]:
df_train=pd.concat([kdd_train,kdd_label],axis=1)

In [558]:
df_train.head()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH,attack,label_2,label_5,Class,Anomaly
0,0.0,3.558064e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal,1,1
1,0.0,1.057999e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal,1,-1
2,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,neptune,attack,DoS,-1,-1
3,0.0,1.681203e-07,6.223962e-06,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal,1,1
4,0.0,1.442067e-07,3.206260e-07,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal,1,-1


In [534]:
##attack=input_file.split(".")[0] +"_IF_attack.csv"
#normal=input_file.split(".")[0] +"_IF_normal.csv"
data_file=train_file.split(".")[0] +"_IF.csv"

In [535]:
print(data_file)

kdd_train_prep2_IF.csv


In [536]:
df_train.to_csv(data_file,index=False)